# CPR outcome grader (Colab)

Replays your Views over past dates with the screener's own backtest code and writes a CSV of
**signal features + outcome (pass / fail / not-triggered)** — the training data for a ranker.

Binance's live API is blocked from Colab, so this uses Binance's public data files instead.
Results and downloaded data are kept in your Google Drive (`MyDrive/cpr-grader`), so a
disconnect only costs you a re-run: the grader resumes where it stopped.

Run the cells top to bottom. Change the settings in the next cell if you like.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────
REPO   = "krivengokul/Crypto-CPR-Screener"
BRANCH = "feat/offline-grader"   # change to "main" once the branch is merged
DAYS   = 90                      # entry dates to grade
TOP    = 50                      # top-N symbols by volume (BTC, ETH, XAUT are always added if listed)

In [ ]:
# ── Drive (keeps data + results between sessions) ───────────────────────
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/cpr-grader"
os.makedirs(BASE, exist_ok=True)
DATA, OUT = f"{BASE}/data", f"{BASE}/out"
print("Saving to", BASE)

In [ ]:
# ── Get the code ────────────────────────────────────────────────────────
import getpass, subprocess, shutil
shutil.rmtree("/content/repo", ignore_errors=True)
url = f"https://github.com/{REPO}.git"
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/repo"])
if r.returncode != 0:
    tok = getpass.getpass("Clone failed - if the repo is private, paste a READ-ONLY GitHub token (blank to stop): ").strip()
    assert tok, "Stopped."
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://{tok}@github.com/{REPO}.git", "/content/repo"], check=True)
!node --version

In [ ]:
# ── 1) Download candles from Binance's public data files ────────────────
# Takes a few minutes. Safe to re-run.
!python /content/repo/tools/offline-grader/download_klines.py --out "{DATA}" --days {DAYS} --top {TOP}

In [ ]:
# ── 2) Grade the Views (resumable) ──────────────────────────────────────
%cd /content/repo
!npx --yes tsx tools/offline-grader/run.ts --data "{DATA}" --out "{OUT}" --days {DAYS}

In [ ]:
# ── 3) Look at the results ──────────────────────────────────────────────
import pandas as pd, json
df = pd.read_csv(f"{OUT}/grades.csv")
print(len(df), "signals,", df.shape[1], "columns")
print(df["result"].value_counts())
graded = df[df["result"].isin(["pass", "fail"])]
print(graded.groupby("view_label")["result"].apply(lambda s: round((s == "pass").mean(), 3)).sort_values(ascending=False))

**Next:** download `grades.csv` from Drive (`MyDrive/cpr-grader/out`) or keep it there for the training notebook.
Columns starting with `f_` are the features known at signal time; columns starting with `leak_` are only
known after the signal day and **must not** be used as model inputs.